In [12]:
pip install gnews

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.7 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=261192abbe480bf0220aaff4847f8741b5a2bcb37c15a2e55d331cf9be985c10
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k
Note: you may need to restart the kernel to use updated packages.


### Monthly News Extraction Without Description

In [17]:
import sys
import os
import warnings
import logging
from gnews import GNews
import pandas as pd
import time
import random  
from tqdm import tqdm
from datetime import datetime
from dateutil.relativedelta import relativedelta

warnings.filterwarnings("ignore") 
logging.getLogger().setLevel(logging.CRITICAL) 

class SuppressOutput:
    def __enter__(self):
        self._original_stdout = sys.stdout
        self._original_stderr = sys.stderr
        sys.stdout = open(os.devnull, 'w')
        sys.stderr = open(os.devnull, 'w')

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stderr.close()
        sys.stdout = self._original_stdout
        sys.stderr = self._original_stderr


search_topics = {
    'RELIANCE': 'Reliance Industries',
    'HDFC': 'HDFC Bank',
    'NIFTY': 'Nifty 50',
    'MARKET': 'Indian Stock Market',
    'SENSEX': 'Sensex',        
    'SP500': 'S&P 500',         
    'NASDAQ': 'NASDAQ Composite',
    'RUT': 'Russell 2000'
}

start_date = datetime(2021, 1, 1)
end_date = datetime(2026, 1, 10)

def fetch_monthly_news(topic, start_dt, end_dt):
    with SuppressOutput():
        try:
           
            google_news = GNews(language='en', country='IN', 
                                start_date=start_dt, end_date=end_dt, 
                                max_results=100) 
            json_resp = google_news.get_news(topic)
            clean_data = []
            for article in json_resp:
                clean_data.append({
                    'Date': article.get('published date'),
                    'Title': article.get('title'),
                    'Link': article.get('url'),
                    'Publisher': article.get('publisher', {}).get('title')
                })
            return clean_data
        except Exception:
            return []

def save_intermediate(data, code):
    """Helper to save data during the loop"""
    if data:
        df = pd.DataFrame(data)
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df.drop_duplicates(subset=['Title', 'Date'], inplace=True)
        filename = f"BIG_DATA_{code}_PARTIAL.csv"
        df.to_csv(filename, index=False)


for code, topic in search_topics.items():
    all_articles = []
    current_date = start_date
    consecutive_fails = 0 
    
    total_months = (end_date.year - start_date.year) * 12 + end_date.month - start_date.month
    
    print(f"\n🚀 Starting: {topic}")
    
    with tqdm(total=total_months, desc=f"Scraping {code}", file=sys.stdout) as pbar:
        while current_date < end_date:
            next_month = current_date + relativedelta(months=1)
            if next_month > end_date:
                next_month = end_date
                
            articles = fetch_monthly_news(topic, current_date, next_month)
            
          
            if articles:
                all_articles.extend(articles)
                consecutive_fails = 0 
            else:
                consecutive_fails += 1
            
            if consecutive_fails >= 5:
                pbar.write(f"⚠️ Pausing {code}: 5 empty months. Moving to next topic to save quota.")
                break 

            if len(all_articles) > 0 and len(all_articles) % 100 == 0:
                save_intermediate(all_articles, code)

            current_date = next_month
            pbar.update(1)
            time.sleep(random.uniform(3.0, 7.0)) 

    if all_articles:
        df = pd.DataFrame(all_articles)
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df.drop_duplicates(subset=['Title', 'Date'], inplace=True)
        
        filename = f"BIG_DATA_{code}_FINAL.csv"
        df.to_csv(filename, index=False)
        print(f"✅ Finished {code}: Saved {len(df)} articles.")
    else:
        print(f"❌ Failed {code}: Zero articles found (Likely Blocked).")

    print("Cooling down for 10 seconds...")
    time.sleep(10)


🚀 Starting: Reliance Industries
Scraping RELIANCE: 61it [16:16, 16.01s/it]                        
✅ Finished RELIANCE: Saved 5526 articles.
Cooling down for 10 seconds...

🚀 Starting: HDFC Bank
Scraping HDFC: 61it [15:22, 15.13s/it]                        
✅ Finished HDFC: Saved 5172 articles.
Cooling down for 10 seconds...

🚀 Starting: Nifty 50
Scraping NIFTY: 61it [13:13, 13.01s/it]                        
✅ Finished NIFTY: Saved 4047 articles.
Cooling down for 10 seconds...

🚀 Starting: Indian Stock Market
Scraping MARKET: 61it [15:40, 15.42s/it]                        
✅ Finished MARKET: Saved 5373 articles.
Cooling down for 10 seconds...

🚀 Starting: Sensex
Scraping SENSEX: 61it [14:58, 14.73s/it]                        
✅ Finished SENSEX: Saved 4862 articles.
Cooling down for 10 seconds...

🚀 Starting: S&P 500
Scraping NASDAQ: 61it [11:39, 11.46s/it]                        
✅ Finished NASDAQ: Saved 3488 articles.
Cooling down for 10 seconds...

🚀 Starting: Russell 2000
Scraping

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv('/kaggle/working/HDFC_News_2021_2025.csv')

In [ ]:
df.head(10)

In [ ]:
!rm  /kaggle/working/HDFC_News_2021_2025.csv

In [ ]:
!rm /kaggle/working/RELIANCE_News_2021_2025.csv

In [ ]:
from gnews import GNews
print("Testing connection to Google News...")
google_news = GNews(language='en', country='IN', start_date=(2024, 1, 1), end_date=(2024, 2, 1))
json_resp = google_news.get_news('Reliance Industries')

if json_resp:
    print(f"✅ Connection OK! Found {len(json_resp)} articles.")
else:
    print("❌ Connection Blocked. Google is returning 0 results.")

In [ ]:
!rm /kaggle/working/BIG_DATA_SP500_PARTIAL.csv

### Weekly Window with Discriptions and Checkpoints

In [ ]:
import sys
import os
import warnings
import logging
from gnews import GNews
import pandas as pd
import time
import random 
from tqdm import tqdm
from datetime import datetime, timedelta

warnings.filterwarnings("ignore") 
logging.getLogger().setLevel(logging.CRITICAL) 

class SuppressOutput:
    def __enter__(self):
        self._original_stdout = sys.stdout
        self._original_stderr = sys.stderr
        sys.stdout = open(os.devnull, 'w')
        sys.stderr = open(os.devnull, 'w')

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stderr.close()
        sys.stdout = self._original_stdout
        sys.stderr = self._original_stderr

search_topics = {
    'RELIANCE': 'Reliance Industries',
    'HDFC': 'HDFC Bank',
    'NIFTY': 'Nifty 50',
    'MARKET': 'Indian Stock Market',
    'SENSEX': 'Sensex',        
    'SP500': 'S&P 500',         
    'NASDAQ': 'NASDAQ Composite',
    'RUT': 'Russell 2000'
}

GLOBAL_START = datetime(2021, 1, 1)
GLOBAL_END = datetime(2026, 1, 10)

def fetch_weekly_news(topic, start_dt, end_dt):
    with SuppressOutput():
        try:
            google_news = GNews(language='en', country='IN', 
                                start_date=start_dt, end_date=end_dt, 
                                max_results=100) 
            json_resp = google_news.get_news(topic)
            clean_data = []
            for article in json_resp:
                clean_data.append({
                    'Date': article.get('published date'),
                    'Title': article.get('title'),
                    'Description': article.get('description'), 
                    'Link': article.get('url'),
                    'Publisher': article.get('publisher', {}).get('title')
                })
            return clean_data
        except Exception:
            return []

def save_checkpoint(data, filename):
    if data:
        df = pd.DataFrame(data)
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df.drop_duplicates(subset=['Title', 'Date'], inplace=True)
        df.to_csv(filename, index=False)

for code, topic in search_topics.items():
    
    partial_filename = f"WEEKLY_DATA_{code}_PARTIAL.csv"
    final_filename = f"WEEKLY_DATA_{code}_Weekly.csv"
    
    all_articles = []
    current_date = GLOBAL_START

    if os.path.exists(partial_filename):
        print(f"\n🔄 Found checkpoint for {code}. Resuming...")
        try:
            existing_df = pd.read_csv(partial_filename)
            existing_df['Date'] = pd.to_datetime(existing_df['Date'], errors='coerce')
            all_articles = existing_df.to_dict('records')
            
            last_date_scraped = existing_df['Date'].max()
            if pd.notnull(last_date_scraped):
                current_date = last_date_scraped + timedelta(days=1)
                if current_date > GLOBAL_END: current_date = GLOBAL_END
                print(f"   ➡ Resuming from: {current_date.date()} (Loaded {len(all_articles)} articles)")
            else:
                print("   ⚠️ Checkpoint empty. Starting fresh.")
        except:
            print("   ❌ Error reading checkpoint. Starting fresh.")
    else:
        print(f"\n🚀 Starting fresh: {code}")

    if current_date >= GLOBAL_END:
        print(f"   ✅ {code} already complete! Skipping.")
        continue

    total_days = (GLOBAL_END - current_date).days
    total_weeks = max(1, total_days // 7)
    consecutive_fails = 0

    with tqdm(total=total_weeks, desc=f"Scraping {code}", file=sys.stdout) as pbar:
        while current_date < GLOBAL_END:
            
            next_week = current_date + timedelta(weeks=1)
            if next_week > GLOBAL_END: next_week = GLOBAL_END
                
            articles = fetch_weekly_news(topic, current_date, next_week)
            
            if articles:
                all_articles.extend(articles)
                consecutive_fails = 0 
            else:
                consecutive_fails += 1

            if consecutive_fails >= 10:
                pbar.write(f"⚠️ Pausing {code}: 10 empty weeks. Moving to next topic.")
                break 

            if len(all_articles) > 0 and len(all_articles) % 500 == 0:
                save_checkpoint(all_articles, partial_filename)

            current_date = next_week
            pbar.update(1)

            pbar.set_postfix(articles=len(all_articles))
            
            time.sleep(random.uniform(4.0, 8.0)) 

    if all_articles:
        save_checkpoint(all_articles, final_filename)
        print(f"✅ Finished {code}: Saved {len(all_articles)} articles.")
        if os.path.exists(partial_filename):
            os.remove(partial_filename)
    else:
        print(f"❌ Failed {code}: Zero articles found.")
        
    print("Cooling down for 30 seconds...")
    time.sleep(30)

In [2]:
import os
import glob

files_to_fix=glob.glob('WEEKLY_DATA_*')

print(len(files_to_fix))

for old_name in files_to_fix:
    new_name=old_name.replace('_.csv','.csv')
    os.rename(old_name,new_name)

print('Job Done')

8
Job Done


In [10]:
import zipfile

files_to_zip=glob.glob('WEEKLY_DATA*')

output_zip_name='Weekly_data.zip'

with zipfile.ZipFile(output_zip_name,'w',zipfile.ZIP_DEFLATED) as files:
    for file in files_to_zip:
        files.write(file)
        print(f'File added named as {file} to zip')

print('Job Done')

File added named as WEEKLY_DATA_SENSEX.csv to zip
File added named as WEEKLY_DATA_RUT.csv to zip
File added named as WEEKLY_DATA_MARKET.csv to zip
File added named as WEEKLY_DATA_RELIANCE.csv to zip
File added named as WEEKLY_DATA_HDFC.csv to zip
File added named as WEEKLY_DATA_NASDAQ.csv to zip
File added named as WEEKLY_DATA_SP500.csv to zip
File added named as WEEKLY_DATA_NIFTY.csv to zip
Job Done


In [3]:
import zipfile

files_to_zip=glob.glob('BIG*')

output_zip_name='Monthly-Data.zip'

print(len(files_to_zip))

with zipfile.ZipFile(output_zip_name,'w',zipfile.ZIP_DEFLATED) as zipf:
    for file in files_to_zip:
        zipf.write(file)
        print(f'File added named as {file} to zip')

print('Job Done')

8
File added named as BIG_DATA_RELIANCE_FINAL.csv to zip
File added named as BIG_DATA_SP500_FINAL.csv to zip
File added named as BIG_DATA_RUT_FINAL.csv to zip
File added named as BIG_DATA_SENSEX_FINAL.csv to zip
File added named as BIG_DATA_HDFC_FINAL.csv to zip
File added named as BIG_DATA_NASDAQ_FINAL.csv to zip
File added named as BIG_DATA_MARKET_FINAL.csv to zip
File added named as BIG_DATA_NIFTY_FINAL.csv to zip
Job Done


In [2]:
import glob
import os

files_to_delete=glob.glob('*_PARTIAL.csv')

for file in files_to_delete:
    !rm {file}

In [4]:
!rm /kaggle/working/Monthly_data

In [ ]:
import requests
import pandas as pd
import os
from datetime import datetime, timedelta
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
API_KEY = user_secrets.get_secret("News_API")

search_topics = {
    'RELIANCE': 'Reliance Industries',
    'HDFC': 'HDFC Bank',
    'NIFTY': 'Nifty 50',
    'MARKET': 'Indian Stock Market',
    'SENSEX': 'Sensex',
    'SP500': 'S&P 500',
    'NASDAQ': 'NASDAQ Composite',
    'RUT': 'Russell 2000'
}

def fetch_daily_news(query, api_key):
    url = "https://newsapi.org/v2/everything"

    from_date = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
    
    params = {
        'q': query,
        'from': from_date,
        'sortBy': 'publishedAt',
        'language': 'en',
        'apiKey': api_key,
        'pageSize': 100 
    }
    
    try:
        response = requests.get(url, params=params)
        data = response.json()
        
        if data.get('status') != 'ok':
            print(f"  ❌ API Error: {data.get('message')}")
            return []
            
        articles = data.get('articles', [])
        formatted_data = []
        
        for article in articles:
            formatted_data.append({
                'Date': article['publishedAt'], 
                'Title': article['title'],
                'Description': article['description'],
                'Link': article['url'],
                'Publisher': article['source']['name']
            })
            
        return formatted_data
        
    except Exception as e:
        print(f"  ❌ Connection Error: {e}")
        return []

print(f"📅 Running Daily Update: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")

for code, query in search_topics.items():
    filename = f"WEEKLY_DATA_{code}.csv"
    print(f"🔄 Updating {code} ({query})...")

    new_data = fetch_daily_news(query, API_KEY)
    
    if not new_data:
        print(f"   ⚠️ No new articles found today (or API limit reached).")
        continue

    new_df = pd.DataFrame(new_data)

    new_df['Date'] = pd.to_datetime(new_df['Date']).dt.strftime('%Y-%m-%d')

    if os.path.exists(filename):
        try:
            existing_df = pd.read_csv(filename)
            # Ensure date column is string for comparison
            existing_df['Date'] = pd.to_datetime(existing_df['Date'], errors='coerce').dt.strftime('%Y-%m-%d')
            
            # Combine
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
            print(f"   📥 Fetched {len(new_df)} new articles.")
        except Exception as e:
            print(f"   ❌ Error reading existing file: {e}. Creating new one.")
            combined_df = new_df
    else:
        print(f"   🆕 File not found. Creating {filename}.")
        combined_df = new_df

    before_dedup = len(combined_df)
    combined_df.drop_duplicates(subset=['Title', 'Date'], inplace=True)
    after_dedup = len(combined_df)
    
    added_count = after_dedup - (len(existing_df) if os.path.exists(filename) else 0)
    combined_df.to_csv(filename, index=False)
    
    if added_count > 0:
        print(f"   ✅ Success! Added {added_count} unique articles. Total: {after_dedup}")
    else:
        print(f"   sz Nothing new added (all duplicates). Total: {after_dedup}")
    
    print("-" * 30)

print("\n🎉 Daily Update Complete!")